# 2 · Measure the gap

This is the moment the study is about. A model trained entirely on renderings
meets real hardware-in-the-loop imagery for the first time.

Nothing here touches training. The real domains are read once, at evaluation, and
have influenced nothing upstream.

Command-line equivalents: `scripts/evaluate.py`, then `scripts/run_conformal.py`.

In [ ]:
# --- environment ------------------------------------------------------------
# Local: nothing to do beyond `pip install -e .` in the repo root.
# Colab: uncomment, then set the two roots below.
#
# from google.colab import drive; drive.mount("/content/drive")
# !pip install -q -e "/content/drive/MyDrive/<path to repo>"

import os

# Where the SPEED+ download and the prepared subsets live.
os.environ.setdefault("SPEEDPLUS_DATA_ROOT", "/path/to/speedplusv2")
# Where checkpoints, predictions and figures go. On Colab put this on the
# mounted Drive so a disconnect does not cost the run.
os.environ.setdefault("SPEEDPLUS_ARTIFACTS", "/path/to/artifacts")

# On Colab, unpack image archives to local disk rather than reading tens of
# thousands of small files across the Drive mount. Set to None when local.
LOCAL_IMAGES = None  # e.g. "/content/images"

In [ ]:
CONFIG = "unfrozen_aug"
SEED = 42
CHECKPOINT = None  # path to the SELECTED checkpoint; None takes the latest

import torch

from speedplus_pose.config import load_config
from speedplus_pose.experiment import load_split_labels
from speedplus_pose.labels import translation_normalisation
from speedplus_pose.model import build_model, resolve_device
from speedplus_pose.normalisation import TranslationNormaliser, constants_for_checkpoint
from speedplus_pose.paths import Paths
from speedplus_pose.train import latest_checkpoint

config = load_config(CONFIG, SEED)
paths = Paths()
device = resolve_device()

checkpoint_path = CHECKPOINT or latest_checkpoint(paths.checkpoints(config.run_name))
checkpoint = torch.load(checkpoint_path, weights_only=False, map_location=device)

model = build_model(config.model, device)
model.load_state_dict(checkpoint["model_state_dict"])

# Prefer the constants stored in the checkpoint, but verify them against the
# training subset. Wrong constants mangle every position prediction in a way
# that still looks entirely plausible.
fallback = translation_normalisation(load_split_labels("train", paths))
normaliser = TranslationNormaliser(*constants_for_checkpoint(checkpoint, *fallback, config.arm), device)

print(f"loaded {checkpoint_path}")

## Is the real data actually harder, or just different?

Before reading any error, rule out the boring explanation: that the real domains
happen to contain harder poses. If the range distributions match, then whatever
difference appears downstream is about **appearance**, not geometry.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from speedplus_pose.experiment import REPORT_DOMAINS

figure, ax = plt.subplots(figsize=(7, 3.2))
for domain in REPORT_DOMAINS:
    labels = load_split_labels(domain, paths)
    ranges = np.linalg.norm(labels[["r1", "r2", "r3"]].to_numpy(), axis=1)
    ax.hist(ranges, bins=40, density=True, histtype="step", linewidth=2, label=domain)
ax.set_xlabel("true range to target [m]")
ax.set_ylabel("density")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

## Predict on all three domains

`calibration` and `in_domain_test` are fresh synthetic splits used later for the
conformal analysis — disjoint from training *and* from the split that selected
this checkpoint.

In [ ]:
from speedplus_pose.evaluate import predict, save_predictions, summarise
from speedplus_pose.experiment import build_eval_dataloader
from speedplus_pose.transforms import build_eval_transform

DOMAINS = [*REPORT_DOMAINS, "calibration", "in_domain_test"]

transform = build_eval_transform(config.image)
predictions = {
    domain: predict(
        model,
        build_eval_dataloader(domain, paths, transform, config.train, local_images=LOCAL_IMAGES),
        normaliser, device, domain,
    )
    for domain in DOMAINS
}
save_predictions(paths.predictions / f"{config.arm}.pkl", predictions)

## The headline

Read `E_R` against the **chance level of 2.207 rad** — the mean angle between two
uniformly random orientations. Absolute rotation errors crowded near that ceiling
understate gaps badly, so *skill above chance* is the honest reading.

In [ ]:
from speedplus_pose.metrics import CHANCE_LEVEL, skill_retained

reference = summarise(predictions["synthetic"])["rotation"]

print(f"{'domain':16} {'E_T':>7} {'E_R':>7} {'S':>7} {'skill':>7} {'kept':>7}")
for domain in REPORT_DOMAINS:
    row = summarise(predictions[domain])
    print(f"{domain:16} {row['translation_relative']:7.3f} {row['rotation']:7.3f} "
          f"{row['speed_score']:7.3f} {CHANCE_LEVEL - row['rotation']:7.3f} "
          f"{skill_retained(reference, row['rotation']):7.0%}")
print(f"\nchance level {CHANCE_LEVEL:.3f} rad")

In [ ]:
# Error distributions, not just means. The real domains do not merely shift -
# they spread, and a mean hides that.
figure, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for domain in REPORT_DOMAINS:
    axes[0].hist(predictions[domain]["translation_relative"], bins=60, range=(0, 1),
                 density=True, histtype="step", linewidth=2, label=domain)
    axes[1].hist(predictions[domain]["rotation"], bins=60, range=(0, np.pi),
                 density=True, histtype="step", linewidth=2, label=domain)
axes[0].set_xlabel("$E_T$ (range-normalised)")
axes[1].set_xlabel("$E_R$ [rad]")
axes[1].axvline(CHANCE_LEVEL, color="black", linestyle="--", linewidth=1)
axes[1].annotate("chance", xy=(CHANCE_LEVEL, 0), xytext=(3, 3),
                 textcoords="offset points", fontsize=8)
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()

## Does the model know it is failing?

Calibrate a conformal region **once**, on fresh synthetic data, then hold it fixed
and measure what it actually covers elsewhere. `q_hat` is never recomputed per
domain — that is the whole experiment.

The gate first: fresh synthetic data is exchangeable with calibration by
construction, so the 90% promise *must* hold there. If it does not, nothing below
it means anything.

In [ ]:
from speedplus_pose.coverage import ALPHA, METRICS, calibrate, gate, measure, scores_from_record

scores = {domain: scores_from_record(record) for domain, record in predictions.items()}
q_hat = calibrate(scores["calibration"], ALPHA)

coverage = {
    domain: measure(q_hat, scores[domain], domain, ALPHA)
    for domain in ("in_domain_test", "lightbox", "sunlamp")
}

passed, z = gate([coverage["in_domain_test"][m] for m in METRICS],
                 n_calibration=len(scores["calibration"]["rotation"]), alpha=ALPHA)
print(f"in-domain gate: {'PASSED' if passed else 'FAILED'}   mean z {np.mean(z):+.2f}")

In [ ]:
from speedplus_pose.conformal import so3_ball_fraction

print(f"q_hat: {q_hat['translation_metres']:.3f} m keep-out sphere, "
      f"{q_hat['rotation']:.3f} rad attitude cone "
      f"({so3_ball_fraction(q_hat['rotation']):.1%} of all orientations)\n")

print(f"{'domain':16} {'translation':>12} {'rotation':>10}")
for domain, entries in coverage.items():
    print(f"{domain:16} {entries['translation_metres'].coverage:11.1%} "
          f"{entries['rotation'].coverage:9.1%}")
print("\nnominal 90%. The guarantee is intact - exchangeability is what broke.")

A region calibrated to 90% covering 9–15% of real rotation errors is the finding:
the model's confidence does not survive the shift, and a wrong pose arrives
wrapped in a confident interval.

Run `scripts/run_conformal.py` for the full table across all eight arms, and
**03 · Reproduce figures** to draw it.